## Cell 1: Setup
**What this demonstrates**: Environment initialisation for RAG Fusion — Anthropic for query variant generation and answer synthesis, OpenAI for embeddings, Chroma for retrieval.
**Expected output**: Setup confirmation with model names and variant count.

In [ ]:
%pip install -q langchain langchain-openai langchain-community chromadb python-dotenv

import os
import time
import pathlib

from dotenv import load_dotenv, find_dotenv
from anthropic import Anthropic
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv(find_dotenv())
_anthropic_key = os.environ.get('ANTHROPIC_API_KEY', '')
_openai_key    = os.environ.get('OPENAI_API_KEY', '')
assert _anthropic_key, 'ANTHROPIC_API_KEY not set — add it to .env'
assert _openai_key,    'OPENAI_API_KEY not set — add it to .env'

# Haiku generates variants cheaply; Sonnet synthesises the final answer
QUERY_MODEL  = 'claude-haiku-4-5-20251001'
ANSWER_MODEL = 'claude-sonnet-4-6'
EMBED_MODEL  = 'text-embedding-3-small'
N_VARIANTS   = 4   # number of query rephrasings to generate
RRF_K        = 60  # standard RRF smoothing constant

client     = Anthropic()
embeddings = OpenAIEmbeddings(model=EMBED_MODEL)

print('Setup complete')
print(f'  Query model  : {QUERY_MODEL}')
print(f'  Answer model : {ANSWER_MODEL}')
print(f'  N variants   : {N_VARIANTS}')
print(f'  RRF k        : {RRF_K}')

## Cell 2: Data — AML and Compliance Corpus
**What this demonstrates**: Loading fintech policy and Basel III documents — a corpus where AML and compliance terminology varies across sections, making it an ideal test case for vocabulary-mismatch retrieval.
**Expected output**: Chunk count and a preview of the terminology variation that RAG Fusion is designed to address.

In [ ]:
_base_candidates = [
    pathlib.Path('../../shared/sample_data'),
    pathlib.Path('shared/sample_data'),
]
BASE_DIR = next((p for p in _base_candidates if p.exists()), None)
assert BASE_DIR is not None, 'Cannot find shared/sample_data'

# Load policy + regulatory docs — both use varied terminology for compliance concepts
doc_files = {
    'fintech_policy': 'fintech_policy.txt',
    'basel_iii'     : 'basel_iii_excerpt.txt',
}

raw_docs: list[Document] = []
for source_name, filename in doc_files.items():
    text = (BASE_DIR / filename).read_text(encoding='utf-8')
    raw_docs.append(Document(page_content=text, metadata={'source': source_name}))
    print(f'  {source_name:<18}: {len(text):,} chars')

splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=60)
chunks: list[Document] = splitter.split_documents(raw_docs)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name='rag_fusion_corpus',
)

print(f'\nCorpus: {len(chunks)} chunks across {len(raw_docs)} documents')
print()
print('Why vocabulary mismatch matters here:')
print('  fintech_policy uses: "suspicious transactions", "reporting obligations"')
print('  Basel III uses    : "disclosure requirements", "regulatory notifications"')
print('  Single query on either misses the other — RAG Fusion covers both')

## Cell 3: Core — Variant Generator, RRF Fusion, RAG Fusion Pipeline
**What this demonstrates**: Four functions implement the full pattern: `generate_variants` rephrases the query, `retrieve_for_variant` runs per-variant retrieval, `rrf_fuse` merges ranked lists, and `rag_fusion` orchestrates the complete pipeline.
**Expected output**: Function definitions confirmed.

In [ ]:
def generate_variants(query: str, n: int = N_VARIANTS) -> list[str]:
    """Generate N rephrasings of the query from different terminological angles.

    Each variant preserves the original information need but uses different
    vocabulary, framing, or regulatory context — exposing documents that
    would be missed by the original phrasing alone.

    Args:
        query: The user's original question.
        n:     Number of variants to generate.

    Returns:
        List of n rephrased queries.
    """
    prompt = '\n'.join([
        f'Generate {n} rephrasings of the following query.',
        'Each rephrasing must:',
        '  1. Preserve the original information need exactly',
        '  2. Use different vocabulary, framing, or regulatory terminology',
        '  3. Be a complete, standalone question',
        'Return one rephrasing per line. No numbering, no prefixes.',
        '',
        f'Query: {query}',
    ])
    response = client.messages.create(
        model=QUERY_MODEL,
        max_tokens=300,
        messages=[{'role': 'user', 'content': prompt}],
    )
    lines = [l.strip() for l in response.content[0].text.strip().splitlines() if l.strip()]
    return lines[:n]


def retrieve_for_variant(variant: str, k: int = 5) -> list[Document]:
    """Retrieve top-k chunks for a single query variant.

    Args:
        variant: A rephrased version of the original query.
        k:       Number of documents to retrieve.

    Returns:
        Ranked list of top-k Document objects.
    """
    return vectorstore.similarity_search(variant, k=k)


def rrf_fuse(
    ranked_lists: list[list[Document]],
    k: int = RRF_K,
) -> list[tuple[Document, float]]:
    """Merge N ranked document lists using Reciprocal Rank Fusion.

    Score formula: score(doc) = sum(1 / (k + rank_i)) across all lists
    where the document appears. The constant k=60 prevents top-ranked
    documents from dominating.

    Args:
        ranked_lists: N lists of documents, each ranked by a separate retrieval.
        k:            RRF smoothing constant (default 60).

    Returns:
        List of (Document, rrf_score) tuples sorted by descending score.
    """
    scores: dict[str, float] = {}
    doc_store: dict[str, Document] = {}

    for ranked_list in ranked_lists:
        for rank, doc in enumerate(ranked_list):
            # Use the first 80 chars as a stable content identifier
            key = doc.page_content[:80]
            scores[key]    = scores.get(key, 0.0) + 1.0 / (k + rank + 1)
            doc_store[key] = doc

    sorted_keys = sorted(scores, key=lambda kk: scores[kk], reverse=True)
    return [(doc_store[kk], scores[kk]) for kk in sorted_keys]


def rag_fusion(
    query: str,
    n_variants: int = N_VARIANTS,
    k_retrieve: int = 5,
) -> dict:
    """Full RAG Fusion pipeline: generate → retrieve × N → fuse → dedup → generate.

    Always includes the original query in the retrieval pool alongside the
    generated variants — the original is never discarded.

    Args:
        query:      The user's original question.
        n_variants: Number of query rephrasings to generate.
        k_retrieve: Documents to retrieve per variant.

    Returns:
        dict with: answer, variants, all_queries, fused_docs, answer.
    """
    t0 = time.perf_counter()

    # Step 1: generate variants
    variants   = generate_variants(query, n=n_variants)
    all_queries = [query] + variants  # original always included

    # Step 2: retrieve for each query (original + variants)
    ranked_lists = [retrieve_for_variant(q, k=k_retrieve) for q in all_queries]

    # Step 3: RRF fusion across all ranked lists
    fused = rrf_fuse(ranked_lists)

    # Step 4: deduplicate — RRF already boosts multi-list docs; don't count twice
    seen: set[str] = set()
    deduped: list[tuple[Document, float]] = []
    for doc, score in fused:
        key = doc.page_content[:80]
        if key not in seen:
            seen.add(key)
            deduped.append((doc, score))

    # Step 5: generate from top-5 fused chunks
    context = '\n\n---\n\n'.join(
        f'[rrf={score:.4f} | source={doc.metadata.get("source","?")}]\n{doc.page_content}'
        for doc, score in deduped[:5]
    )
    response = client.messages.create(
        model=ANSWER_MODEL,
        max_tokens=400,
        system='Answer using only the provided context. Cite specific facts with their source.',
        messages=[{'role': 'user', 'content': f'Context:\n{context}\n\nQuestion: {query}'}],
    )

    return {
        'query'       : query,
        'variants'    : variants,
        'all_queries' : all_queries,
        'ranked_lists': ranked_lists,
        'fused'       : deduped,
        'answer'      : response.content[0].text.strip(),
        'latency_ms'  : (time.perf_counter() - t0) * 1000,
    }


print('RAG Fusion pipeline defined')
print('  generate_variants()  — N rephrasings via Haiku')
print('  retrieve_for_variant() — per-variant similarity search')
print('  rrf_fuse()           — merge ranked lists via RRF')
print('  rag_fusion()         — full pipeline: generate → retrieve × N → fuse → generate')

## Cell 4: Run — Suspicious Transactions Query
**What this demonstrates**: The workshop demo query — AML reporting obligations — run through RAG Fusion. Four variants cover the full AML terminology space; RRF fusion surfaces documents that the original query alone would miss.
**Expected output**: Generated variants, total retrieval count, and the final answer.

In [ ]:
QUERY = 'What are the reporting obligations for suspicious transactions?'

print(f'Query: {QUERY}')
print()

result = rag_fusion(QUERY)

# Show generated variants
print('Generated variants:')
for i, v in enumerate(result['variants'], 1):
    print(f'  {i}. {v}')
print()

# Retrieval summary
total_retrieved = sum(len(rl) for rl in result['ranked_lists'])
print(f'Retrieval: {len(result["all_queries"])} queries × 5 docs = {total_retrieved} total')
print(f'After RRF + dedup: {len(result["fused"])} unique chunks')
print(f'Latency: {result["latency_ms"]:.0f} ms')
print()
print('=' * 65)
print('ANSWER')
print('=' * 65)
print(result['answer'])

## Cell 5: Inspect — Variant Coverage and RRF Scores
**What this demonstrates**: The per-variant retrieval results and RRF score distribution — showing which chunks appeared in multiple variant lists (high RRF score) and which were caught by only one variant (lower score but still included).
**Expected output**: Per-variant top-3 chunk previews, fused ranking with scores, and a baseline comparison showing what single-query retrieval would have missed.

In [ ]:
# ── Per-variant retrieval preview ─────────────────────────────────────────────
print('PER-VARIANT RETRIEVAL (top 3 per query)')
print('=' * 65)
for i, (query_text, ranked_list) in enumerate(zip(result['all_queries'], result['ranked_lists'])):
    label = 'Original' if i == 0 else f'Variant {i}'
    print(f'\n[{label}] {query_text[:60]}')
    for rank, doc in enumerate(ranked_list[:3], 1):
        src     = doc.metadata.get('source', '?')
        preview = doc.page_content[:70].replace('\n', ' ')
        print(f'  {rank}. [{src}] {preview}...')

# ── Fused ranking ─────────────────────────────────────────────────────────────
print()
print('RRF FUSED RANKING (top 10)')
print('=' * 65)
print(f"{'Rank':<5} {'RRF score':<12} {'Source':<18} {'Chunk preview'}")
print('-' * 65)
for rank, (doc, score) in enumerate(result['fused'][:10], 1):
    src     = doc.metadata.get('source', '?')
    preview = doc.page_content[:40].replace('\n', ' ')
    print(f'{rank:<5} {score:<12.5f} {src:<18} {preview}...')

# ── Baseline comparison ───────────────────────────────────────────────────────
print()
print('BASELINE: what single-query retrieval returns')
print('=' * 65)
baseline = retrieve_for_variant(QUERY, k=5)
baseline_keys = {d.page_content[:80] for d in baseline}
fused_keys    = {d.page_content[:80] for d, _ in result['fused'][:5]}
new_in_fusion = fused_keys - baseline_keys

print(f'Baseline top-5 keys : {len(baseline_keys)}')
print(f'Fused top-5 keys    : {len(fused_keys)}')
print(f'New chunks in fusion: {len(new_in_fusion)}')
print()
if new_in_fusion:
    print('Chunks caught by RAG Fusion that single-query missed:')
    for key in new_in_fusion:
        print(f'  → {key[:80]}...')
else:
    print('All top-5 fused chunks also appeared in baseline.')
    print('(This can happen when the corpus is small — effect is stronger on larger corpora)')

## Cell 6: Fintech — Market Research with Perspective Diversity
**What this demonstrates**: RAG Fusion on a market research query where perspective diversity matters — a credit risk question that benefits from variants covering different angles (counterparty risk, default exposure, portfolio concentration).
**Expected output**: Variants covering different credit risk angles, fused results, and a more complete answer than single-query RAG.

In [ ]:
MARKET_QUERY = 'What are the credit risk exposure limits in our lending policy?'

print('MARKET RESEARCH — PERSPECTIVE DIVERSITY DEMO')
print(f'Query: {MARKET_QUERY}')
print()

market_result = rag_fusion(MARKET_QUERY)

print('Generated variants (different credit risk angles):')
for i, v in enumerate(market_result['variants'], 1):
    print(f'  {i}. {v}')
print()

# Show which sources each variant drew from
print('Source coverage per variant:')
for i, (query_text, ranked_list) in enumerate(zip(market_result['all_queries'], market_result['ranked_lists'])):
    label   = 'Original' if i == 0 else f'Variant {i}'
    sources = list({d.metadata.get('source', '?') for d in ranked_list})
    print(f'  [{label:<12}] sources: {", ".join(sorted(sources))}')

print()
print(f'Fused unique chunks: {len(market_result["fused"])}')
print(f'Latency: {market_result["latency_ms"]:.0f} ms')
print()
print('=' * 65)
print('CREDIT RISK ANALYSIS')
print('=' * 65)
print(market_result['answer'])
print()
print('-' * 65)
print('RAG Fusion value here:')
print('  Policy docs use "loan-to-value limits", "concentration risk", "credit exposure"')
print('  Each variant targets a different synonym cluster')
print('  RRF fusion surfaces the most relevant chunk regardless of which term it uses')